# PDF Vietnamese Translator - Google Colab

Công cụ dịch PDF sang tiếng Việt giữ nguyên layout, chạy trên Google Colab.

## Tính năng:
- Trích xuất văn bản và layout từ PDF
- Dịch thuật sử dụng LLM (OpenAI, DeepSeek, Gemini...)
- Giữ nguyên định dạng, font chữ, vị trí văn bản
- Hỗ trợ công thức toán học và thuật ngữ kỹ thuật

## 1. Cài đặt thư viện cần thiết

In [ ]:
!pip install -q pymupdf reportlab pillow openai tqdm numpy

print("✅ Đã cài đặt xong các thư viện")

## 2. Tạo cấu trúc dự án

In [ ]:
import os

# Tạo cấu trúc thư mục
os.makedirs('/content/pdf-viet-translator/src', exist_ok=True)
print("✅ Đã tạo cấu trúc dự án")

## 3. Copy code vào các file Python

In [ ]:
%%writefile /content/pdf-viet-translator/src/pdf_extractor.py
"""
PDF Text and Layout Extractor
Extracts text blocks with positioning information from PDF files.
"""

import fitz  # PyMuPDF
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple


@dataclass
class TextBlock:
    """Represents a text block with position and style information."""
    text: str
    bbox: Tuple[float, float, float, float]  # (x0, y0, x1, y1)
    page_num: int
    font_name: str
    font_size: float
    color: int
    is_bold: bool
    is_italic: bool
    block_type: str


@dataclass
class PageLayout:
    """Layout information for a page."""
    page_num: int
    width: float
    height: float
    text_blocks: List[TextBlock]
    images: List[Dict[str, Any]]


class PDFExtractor:
    """Extract text and layout from PDF files."""

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.doc = fitz.open(pdf_path)

    def close(self):
        if self.doc:
            self.doc.close()

    def get_page_count(self) -> int:
        return len(self.doc)

    def extract_page_layout(self, page_num: int) -> PageLayout:
        """Extract layout information from a single page."""
        page = self.doc[page_num]
        width = page.rect.width
        height = page.rect.height

        text_blocks = self._extract_text_blocks(page, page_num)
        images = self._extract_images(page, page_num)

        return PageLayout(
            page_num=page_num,
            width=width,
            height=height,
            text_blocks=text_blocks,
            images=images
        )

    def _extract_text_blocks(self, page, page_num: int) -> List[TextBlock]:
        """Extract text blocks with formatting information."""
        blocks = []
        blocks_raw = page.get_text("dict")["blocks"]

        for block in blocks_raw:
            if block["type"] != 0:  # Skip non-text blocks
                continue

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span.get("text", "").strip()
                    if not text:
                        continue

                    bbox = span["bbox"]
                    font_name = span.get("font", "helv")
                    font_size = span.get("size", 12)
                    color = span.get("color", 0)
                    flags = span.get("flags", 0)

                    is_bold = bool(flags & 2**4)
                    is_italic = bool(flags & 2**1)

                    block_type = self._classify_block_type(font_size, is_bold, bbox, page.rect.height)

                    text_block = TextBlock(
                        text=text,
                        bbox=(bbox[0], bbox[1], bbox[2], bbox[3]),
                        page_num=page_num,
                        font_name=font_name,
                        font_size=font_size,
                        color=color,
                        is_bold=is_bold,
                        is_italic=is_italic,
                        block_type=block_type
                    )
                    blocks.append(text_block)

        return blocks

    def _classify_block_type(self, font_size: float, is_bold: bool, bbox: tuple, page_height: float) -> str:
        """Classify text block type based on formatting."""
        y_pos = bbox[1]

        if font_size > 18 or (is_bold and font_size > 14):
            return "title"
        elif y_pos < page_height * 0.1:
            return "header"
        elif y_pos > page_height * 0.9:
            return "footer"
        elif font_size < 10:
            return "footnote"
        else:
            return "text"

    def _extract_images(self, page, page_num: int) -> List[Dict[str, Any]]:
        """Extract image information from page."""
        images = []
        image_list = page.get_images(full=True)

        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = self.doc.extract_image(xref)
            if base_image:
                images.append({
                    "page_num": page_num,
                    "xref": xref,
                    "width": base_image.get("width", 0),
                    "height": base_image.get("height", 0),
                    "bbox": img[1] if len(img) > 1 else None
                })

        return images

    def extract_all_layouts(self) -> List[PageLayout]:
        """Extract layout from all pages."""
        layouts = []
        for page_num in range(len(self.doc)):
            layouts.append(self.extract_page_layout(page_num))
        return layouts

In [ ]:
%%writefile /content/pdf-viet-translator/src/translator.py
"""
Translation Module with LLM Support
Supports OpenAI-compatible APIs (OpenAI, DeepSeek, Gemini, etc.)
"""

import os
import json
import time
from typing import List, Dict, Any, Optional
from openai import OpenAI
from dataclasses import dataclass


@dataclass
class TranslationConfig:
    """Configuration for translation."""
    api_key: str
    base_url: str = "https://api.openai.com/v1"
    model: str = "gpt-4o-mini"
    source_lang: str = "English"
    target_lang: str = "Vietnamese"
    batch_size: int = 5
    max_retries: int = 3
    temperature: float = 0.3


class Translator:
    """Handles text translation using LLM APIs."""

    def __init__(self, config: TranslationConfig):
        self.config = config
        self.client = OpenAI(
            api_key=config.api_key,
            base_url=config.base_url if config.base_url else None
        )

    def translate_text(self, text: str, context: Optional[str] = None) -> str:
        """Translate a single text block."""
        if not text.strip():
            return text

        prompt = self._build_translation_prompt(text, context)

        for attempt in range(self.config.max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.config.model,
                    messages=[
                        {"role": "system", "content": self._get_system_prompt()},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=self.config.temperature,
                    max_tokens=2000
                )
                translated = response.choices[0].message.content.strip()
                return self._clean_translation(translated, text)

            except Exception as e:
                if attempt < self.config.max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    print(f"Translation failed after {self.config.max_retries} attempts: {e}")
                    return text

        return text

    def translate_batch(self, texts: List[str], contexts: Optional[List[str]] = None) -> List[str]:
        """Translate a batch of texts."""
        results = []
        for i, text in enumerate(texts):
            context = contexts[i] if contexts and i < len(contexts) else None
            translated = self.translate_text(text, context)
            results.append(translated)
        return results

    def _get_system_prompt(self) -> str:
        """Get system prompt for translation."""
        return f"""You are a professional translator specializing in translating {self.config.source_lang} documents to {self.config.target_lang}.

Rules:
1. Preserve technical terms, names, and code snippets exactly as they appear
2. Maintain the original formatting and structure
3. Keep mathematical formulas, variables, and technical notation unchanged
4. Translate naturally and accurately to {self.config.target_lang}
5. Return ONLY the translated text without explanations"""

    def _build_translation_prompt(self, text: str, context: Optional[str] = None) -> str:
        """Build translation prompt."""
        prompt = f"Translate the following text to {self.config.target_lang}:\n\n{text}"

        if context:
            prompt += f"\n\nContext (surrounding text):\n{context}"

        prompt += "\n\nTranslation:"
        return prompt

    def _clean_translation(self, translated: str, original: str) -> str:
        """Clean up translation output."""
        prefixes = ["Translation:", "Dịch:", "Kết quả:"]
        for prefix in prefixes:
            if translated.startswith(prefix):
                translated = translated[len(prefix):].strip()

        if translated.startswith('"') and translated.endswith('"'):
            translated = translated[1:-1]

        return translated

In [ ]:
%%writefile /content/pdf-viet-translator/src/renderer.py
"""
PDF Renderer - Creates new PDF with translated text preserving layout.
"""

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import point
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.colors import Color
from typing import List, Dict, Any, Tuple
import fitz  # PyMuPDF for reading original PDF
from src.pdf_extractor import PageLayout, TextBlock


class PDFRenderer:
    """Renders translated text into a new PDF with preserved layout."""

    def __init__(self, output_path: str, font_name: str = "Helvetica"):
        self.output_path = output_path
        self.font_name = font_name

    def render_pdf(self, original_pdf_path: str, page_layouts: List[PageLayout],
                   translations: Dict[Tuple[int, int], str]):
        """Render translated PDF."""
        doc = fitz.open(original_pdf_path)
        c = None

        try:
            first_page = doc[0]
            page_width = first_page.rect.width
            page_height = first_page.rect.height

            c = canvas.Canvas(self.output_path, pagesize=(page_width, page_height))

            for page_num, layout in enumerate(page_layouts):
                self._render_page(c, layout, translations, page_num, doc)
                c.showPage()

            c.save()

        finally:
            if c:
                c.save()
            doc.close()

    def _render_page(self, c, layout: PageLayout, translations: Dict[Tuple[int, int], str],
                     page_num: int, original_doc):
        """Render a single page."""
        # Draw original page as background
        original_page = original_doc[page_num]

        # Render text blocks
        for block_idx, block in enumerate(layout.text_blocks):
            translation_key = (page_num, block_idx)
            text = translations.get(translation_key, block.text)

            if not text.strip():
                continue

            self._render_text_block(c, block, text)

    def _render_text_block(self, c, block: TextBlock, text: str):
        """Render a text block at its original position."""
        # Set font
        font_name = self._get_safe_font(block)
        font_size = block.font_size

        # Calculate position
        x, y = block.bbox[0], block.bbox[1]

        # Adjust y position (ReportLab uses bottom-left origin)
        y_adjusted = block.bbox[3] - font_size * 0.2

        # Set text color
        color = self._parse_color(block.color)
        c.setFillColor(color)

        # Set font
        c.setFont(font_name, font_size)

        # Handle text wrapping within original bounding box
        max_width = block.bbox[2] - block.bbox[0]
        self._draw_wrapped_text(c, text, x, y_adjusted, max_width, font_size)

    def _get_safe_font(self, block: TextBlock) -> str:
        """Get a font that supports Vietnamese characters."""
        if block.is_bold and block.is_italic:
            return "Helvetica-BoldOblique"
        elif block.is_bold:
            return "Helvetica-Bold"
        elif block.is_italic:
            return "Helvetica-Oblique"
        else:
            return "Helvetica"

    def _parse_color(self, color_int: int) -> Color:
        """Convert integer color to ReportLab Color."""
        r = ((color_int >> 16) & 0xFF) / 255.0
        g = ((color_int >> 8) & 0xFF) / 255.0
        b = (color_int & 0xFF) / 255.0
        return Color(r, g, b)

    def _draw_wrapped_text(self, c, text: str, x: float, y: float,
                          max_width: float, font_size: float):
        """Draw text with wrapping to fit within bounding box."""
        words = text.split()
        lines = []
        current_line = []
        current_width = 0

        for word in words:
            word_width = c.stringWidth(word + " ", self.font_name, font_size)

            if current_width + word_width <= max_width or not current_line:
                current_line.append(word)
                current_width += word_width
            else:
                lines.append(" ".join(current_line))
                current_line = [word]
                current_width = word_width

        if current_line:
            lines.append(" ".join(current_line))

        # Draw lines
        line_height = font_size * 1.2
        for i, line in enumerate(lines):
            c.drawString(x, y - i * line_height, line)

In [ ]:
%%writefile /content/pdf-viet-translator/src/pipeline.py
"""
Main pipeline for PDF translation to Vietnamese.
Orchestrates extraction, translation, and rendering.
"""

import os
import json
from typing import List, Dict, Tuple, Optional
from tqdm import tqdm
from src.pdf_extractor import PDFExtractor, PageLayout, TextBlock
from src.translator import Translator, TranslationConfig
from src.renderer import PDFRenderer


class PDFTranslationPipeline:
    """Complete pipeline for translating PDFs to Vietnamese."""

    def __init__(self, config: TranslationConfig):
        self.config = config
        self.translator = Translator(config)
        self.extractor = None
        self.renderer = None

    def translate_pdf(self, input_pdf: str, output_pdf: str,
                     pages: Optional[List[int]] = None) -> Dict[str, Any]:
        """
        Translate a PDF file to Vietnamese.

        Args:
            input_pdf: Path to input PDF
            output_pdf: Path to output PDF
            pages: Optional list of page numbers to translate (0-indexed)

        Returns:
            Dictionary with translation statistics
        """
        print(f"Starting translation of {input_pdf}")

        # Extract
        print("Step 1: Extracting text and layout...")
        self.extractor = PDFExtractor(input_pdf)
        layouts = self._extract_layouts(pages)
        print(f"Extracted {len(layouts)} pages")

        # Collect text blocks for translation
        print("Step 2: Collecting text blocks...")
        text_blocks = self._collect_text_blocks(layouts)
        print(f"Found {len(text_blocks)} text blocks to translate")

        # Translate
        print("Step 3: Translating to Vietnamese...")
        translations = self._translate_blocks(text_blocks, layouts)
        print(f"Translated {len(translations)} blocks")

        # Render
        print("Step 4: Rendering translated PDF...")
        self.renderer = PDFRenderer(output_pdf)
        self.renderer.render_pdf(input_pdf, layouts, translations)

        # Close extractor
        if self.extractor:
            self.extractor.close()

        stats = {
            "input_pdf": input_pdf,
            "output_pdf": output_pdf,
            "pages_processed": len(layouts),
            "blocks_translated": len(translations),
            "target_language": self.config.target_lang
        }

        print(f"\n✅ Translation complete! Output saved to: {output_pdf}")
        return stats

    def _extract_layouts(self, pages: Optional[List[int]]) -> List[PageLayout]:
        """Extract layouts from specified pages or all pages."""
        if pages is not None:
            layouts = []
            for page_num in pages:
                layouts.append(self.extractor.extract_page_layout(page_num))
            return layouts
        else:
            return self.extractor.extract_all_layouts()

    def _collect_text_blocks(self, layouts: List[PageLayout]) -> List[Tuple[int, int, TextBlock]]:
        """Collect all translatable text blocks with their positions."""
        blocks = []
        for layout in layouts:
            for block_idx, block in enumerate(layout.text_blocks):
                text = block.text.strip()
                if text and len(text) > 1:  # Skip very short texts
                    blocks.append((layout.page_num, block_idx, block))
        return blocks

    def _translate_blocks(self, text_blocks: List[Tuple[int, int, TextBlock]],
                          layouts: List[PageLayout]) -> Dict[Tuple[int, int], str]:
        """Translate all text blocks."""
        translations = {}
        layout_dict = {layout.page_num: layout for layout in layouts}

        # Prepare batch
        texts_to_translate = []
        block_positions = []

        for page_num, block_idx, block in text_blocks:
            texts_to_translate.append(block.text)
            block_positions.append((page_num, block_idx))

        # Translate in batches
        translated_texts = []
        batch_size = self.config.batch_size

        for i in tqdm(range(0, len(texts_to_translate), batch_size), desc="Translating"):
            batch = texts_to_translate[i:i + batch_size]
            batch_translations = self.translator.translate_batch(batch)
            translated_texts.extend(batch_translations)

        # Map translations back to positions
        for (page_num, block_idx), translated in zip(block_positions, translated_texts):
            translations[(page_num, block_idx)] = translated

        return translations

    def save_translation_map(self, translations: Dict[Tuple[int, int], str],
                             output_path: str):
        """Save translation mapping to JSON for inspection."""
        serializable = {
            f"page_{k[0]}_block_{k[1]}": v
            for k, v in translations.items()
        }
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(serializable, f, ensure_ascii=False, indent=2)

In [ ]:
%%writefile /content/pdf-viet-translator/src/__init__.py
"""
PDF Vietnamese Translator
A simple tool to translate PDF documents to Vietnamese while preserving layout.
"""

from src.pdf_extractor import PDFExtractor
from src.translator import Translator, TranslationConfig
from src.renderer import PDFRenderer
from src.pipeline import PDFTranslationPipeline

__version__ = "1.0.0"

__all__ = [
    'PDFExtractor',
    'Translator',
    'TranslationConfig',
    'PDFRenderer',
    'PDFTranslationPipeline'
]

## 4. Cấu hình API Key

Bạn có thể sử dụng OpenAI, DeepSeek, hoặc bất kỳ API nào tương thích OpenAI.

**Lưu ý**: Thay YOUR_API_KEY bằng API key thật của bạn.

In [ ]:
from google.colab import userdata

# Cách 1: Sử dụng Colab Secrets (khuyên dùng)
# Thêm API key vào Secrets với tên 'OPENAI_API_KEY' hoặc 'DEEPSEEK_API_KEY'
try:
    api_key = userdata.get('OPENAI_API_KEY')  # Hoặc DEEPSEEK_API_KEY
    print("✅ Đã lấy API key từ Secrets")
except:
    # Cách 2: Nhập thủ công
    api_key = input("Nhập API key của bạn: ")

# Cấu hình dịch thuật
from src.translator import TranslationConfig

config = TranslationConfig(
    api_key=api_key,
    base_url="https://api.openai.com/v1",  # Đổi thành "https://api.deepseek.com/v1" nếu dùng DeepSeek
    model="gpt-4o-mini",  # Hoặc "deepseek-chat" cho DeepSeek
    source_lang="English",
    target_lang="Vietnamese",
    batch_size=5,
    temperature=0.3
)

print("✅ Đã cấu hình dịch thuật")

## 5. Upload PDF file

Tải lên file PDF bạn muốn dịch.

In [ ]:
from google.colab import files

# Upload PDF file
print("Chọn file PDF để tải lên...")
uploaded = files.upload()

# Lấy tên file đã upload
if uploaded:
    input_pdf = list(uploaded.keys())[0]
    print(f"✅ Đã tải lên: {input_pdf}")
else:
    print("❌ Chưa tải lên file nào")

## 6. Chạy dịch thuật

Bắt đầu quá trình dịch PDF sang tiếng Việt.

In [ ]:
import sys
sys.path.append('/content/pdf-viet-translator')

from src.pipeline import PDFTranslationPipeline

# Khởi tạo pipeline
pipeline = PDFTranslationPipeline(config)

# Tên file output
output_pdf = 'translated_' + input_pdf

# Chạy dịch thuật
# Để dịch chỉ một số trang, thêm tham số: pages=[0, 1, 2]
stats = pipeline.translate_pdf(
    input_pdf=input_pdf,
    output_pdf=output_pdf,
    # pages=[0, 1, 2]  # Uncomment để dịch chỉ 3 trang đầu
)

print("\n📊 Thống kê:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 7. Tải xuống kết quả

In [ ]:
from google.colab import files

# Tải xuống file PDF đã dịch
print("Đang tải xuống file dịch...")
files.download(output_pdf)
print("✅ Đã tải xuống xong!")

## 8. (Tùy chọn) Lưu bản dịch dưới dạng JSON để xem

In [ ]:
# Lưu mapping dịch thuật để kiểm tra
translation_map_path = 'translation_map.json'
pipeline.save_translation_map(pipeline.translate_pdf.__defaults__, translation_map_path)
print(f"✅ Đã lưu mapping dịch thuật tại: {translation_map_path}")

# Xem trước một vài bản dịch
import json
with open(translation_map_path, 'r', encoding='utf-8') as f:
    translations = json.load(f)

print("\n📝 Xem trước 5 bản dịch đầu tiên:")
for i, (key, value) in enumerate(list(translations.items())[:5]):
    print(f"\n{key}:")
    print(f"  Dịch: {value[:100]}...")

## 9. Sử dụng các API khác (DeepSeek, Gemini, v.v.)

### DeepSeek API:
```python
config = TranslationConfig(
    api_key="your_deepseek_api_key",
    base_url="https://api.deepseek.com/v1",
    model="deepseek-chat",
    source_lang="English",
    target_lang="Vietnamese"
)
```

### Gemini API (thông qua OpenRouter):
```python
config = TranslationConfig(
    api_key="your_openrouter_api_key",
    base_url="https://openrouter.ai/api/v1",
    model="google/gemini-pro",
    source_lang="English",
    target_lang="Vietnamese"
)
```